# Transcript counter

This notebook demonstrates how to 

## Import Required Libraries

Import the necessary libraries for working with zarr files, including zarr, numpy, and pandas.

In [ ]:
import numpy as np
import pandas as pd
import dask.dataframe as dd
import pyarrow as pa
from pathlib import Path
import sys
import os

print(sys.executable)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("PyArrow version:", pa.__version__)
print("PyArrow location:", os.path.dirname(pa.__file__))
print("\nAll dependencies installed successfully!")

## Get Transcript Counts

Note: 10x refers to the transcripts as "feature_name"

1. Open the region summary csv and get the list of cell IDs we're interested in. Ignore any data that starts with a "#"
2. Create a results csv file
3. Read the transcripts parquet file, and read only the columns that are important for the analysis
4. Filter the data by removing any data where the quality score is less than 20 (matches 10x behavior) and remove any "unassigned" or "negcontrol" feature names
5. Create a pivot table of the data based on cell_id and transcript
6. Output the pivot table to the csv document
7. Print out any cell IDs that were found in the region summary, but did not appear in the parquet file.

In [ ]:
def calculate_transcript_counts(top_level_folder, region_subfoler, input_file):
    transcripts_parquet_path =  Path(top_level_folder) / Path(region_subfoler) / "transcripts.parquet"

    # read parquet file with dask
    file_data = dd.read_parquet(transcripts_parquet_path, index=False, engine='pyarrow', columns=['cell_id', 'feature_name', 'qv'])

    unfiltered_count = file_data.shape[0].compute()
    print(f"Total number of transcripts before filtering: {unfiltered_count}")

    # only keep data with a q score of 20 or higher to match 10x defaults
    file_data = file_data[file_data["qv"] >= 20]

    filtered_count = file_data.shape[0].compute()
    print(f"Total number of transcripts after filtering: {filtered_count}")

    # filter the data to only include valid transcripts
    def filter_valid_transcripts(cell_data):
        #filter out feature names that contain "unknown"
        cell_data = cell_data[~cell_data["feature_name"].str.contains("UnassignedCodeword_")]

        #filter out feature names that contain "NegControl"
        cell_data = cell_data[~cell_data["feature_name"].str.contains("NegControl")]
        return cell_data

    # get the list of transcripts 
    fileData_filtered = filter_valid_transcripts(file_data)

    # create a pivot table with cell_id as rows and feature_name as columns, and the count of transcripts as values
    cell_gene_matrix = (
        fileData_filtered
        .groupby(["cell_id", "feature_name"])
        .size()
        .compute()
        .unstack(fill_value=0)
    )

    region_summary = pd.read_csv(input_file, comment='#')

    # filter the cell_gene_matrix to only include cells that are in the region summary file
    valid_cell_ids = region_summary['Cell ID'][region_summary['Cell ID'].isin(cell_gene_matrix.index)]
    # sort the cell_gene_matrix by cell_id 
    results_df = cell_gene_matrix.loc[valid_cell_ids].sort_index()

    # Print which cells were not found 
    missing_cells = region_summary['Cell ID'][~region_summary['Cell ID'].isin(cell_gene_matrix.index)]
    if len(missing_cells) > 0:
        print(f"Warning: {len(missing_cells)} cells not found in parquet data:")
        print(missing_cells.tolist())

    # further filter to only include cells where Th > 3
    print(f"Total number of transcripts before removing Th > 3: {results_df.shape[0]}")
    th_filtered_results_df = results_df[results_df['Th'] > 3]
    print(f"Total number of transcripts after removing Th > 3: {th_filtered_results_df.shape[0]}")

    # save the results to csv
    input_file_name = str(os.path.basename(input_file))

    results_csv_path = os.path.join("output", input_file_name.replace(".csv", "_transcript_counts.csv"))
    results_csv_path_filtered = os.path.join("output", input_file_name.replace(".csv", "_th_filtered_transcript_counts.csv"))
    #delete the file if it already exists
    if os.path.exists(results_csv_path):
        os.remove(results_csv_path)
    results_df.to_csv(results_csv_path)

    if os.path.exists(results_csv_path_filtered):
        os.remove(results_csv_path_filtered)
    th_filtered_results_df.to_csv(results_csv_path_filtered)



## Setup file paths
Set the name of the folder path we should be using to find the associated data files

In [ ]:
topLevelFolder = "20250827__205437__CSU_Bouchet_v1_2025-08-27"

intput_folder = Path("input")
# read all the files from the input folder and print the results
input_files = os.listdir(intput_folder)

for input_file in input_files:
    # get the region name from the input file name
    region_name = input_file.split("_")[1].lower()

    # find the corresponding subfolder name
    subfolder = None
    for folder in os.listdir(topLevelFolder):
        if region_name in folder.lower().replace("-", "").replace("_", ""):
            subfolder = folder
            break
    
    if subfolder is None:
        print(f"Error: No subfolder found for region {region_name} in top level folder {topLevelFolder}")
        continue

    print(f"Processing region {region_name} for input file {input_file}")
    results_df = calculate_transcript_counts(topLevelFolder, subfolder, Path(os.path.join(intput_folder, input_file)))


In [ ]:

# Categorize cells by Slc32a1 (VGAT) and Slc17a6 (vGluT2) expression
def categorize_cell(row):
    has_vgat = row['Slc32a1'] > 0
    has_vglut2 = row['Slc17a6'] > 0
    
    if has_vgat and has_vglut2:
        return 'Both'
    elif has_vgat:
        return 'VGAT'
    elif has_vglut2:
        return 'vGluT2'
    else:
        return 'Neither'

th_positive_cells['Category'] = th_positive_cells.apply(categorize_cell, axis=1)
category_counts = th_positive_cells['Category'].value_counts()

# Create pie chart
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', startangle=90)
plt.title(f'VGAT and vGluT2 Expression in Th+ Cells (n={len(th_positive_cells)})')
plt.show()

print(category_counts)

# Create bar graph showing cell counts by category
plt.figure(figsize=(8, 6))
category_counts.plot(kind='bar', color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
plt.xlabel('Category')
plt.ylabel('Number of Cells')
plt.title(f'Th+ Cell Distribution by Marker Expression (n={len(th_positive_cells)})')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


def categorize_cell_for_transcript(row, transcript):
    has_transcript = row[transcript] > 0

    if has_transcript:
        return transcript
    else:
        return 'No ' + transcript

transcripts_to_count = ['Nr3c1', 'Slc6a3']
for transcript in transcripts_to_count:
    th_positive_cells['Category'] = th_positive_cells.apply(lambda row: categorize_cell_for_transcript(row, transcript), axis=1)
    category_counts = th_positive_cells['Category'].value_counts()

    # Create pie chart
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 6))
    plt.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', startangle=90)
    plt.title(f'{transcript} Expression in Th+ Cells (n={len(th_positive_cells)})')
    plt.show()

    print(category_counts)

# Get Slc6a3 counts for each Th+ cell
slc6a3_counts = th_positive_cells['Slc6a3'].sort_values(ascending=False)

# Create bar chart
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 6))
slc6a3_counts.plot(kind='bar', color='steelblue')
plt.xlabel('Cell ID')
plt.ylabel('Slc6a3 Transcript Count')
plt.title(f'Slc6a3 Expression in Th+ Cells (n={len(th_positive_cells)})')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()